System construction and test


In [2]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [5]:
dataini = '2022-04-03 19:30:00'
datafim = '2023-04-03 19:30:00'

In [7]:
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
display(dftitulosdados)

,symbol,moeda,intervalo,datetime,open,high,low,close,volume
0,GGAL,USD,60min,2022-04-04 09:00:00,8.9836,9.1457,8.9755,9.0079,45436.0
1,GGAL,USD,60min,2022-04-04 10:00:00,9.0484,9.0809,8.9755,8.9917,49030.0
2,GGAL,USD,60min,2022-04-04 11:00:00,8.9836,8.9998,8.9268,8.9268,41129.0
3,GGAL,USD,60min,2022-04-04 12:00:00,8.9268,9.0160,8.9268,8.9512,34248.0
4,GGAL,USD,60min,2022-04-04 13:00:00,8.9593,8.9593,8.8863,8.9187,67152.0
...,...,...,...,...,...,...,...,...,...
2101,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0
2102,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0
2103,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0
2104,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0


In [9]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
dayM = 8  
semM = 4


In [11]:

# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)
display (dfstoch)

,symbol,moeda,intervalo,datetime,open,high,low,close,volume,khigh,dhigh
0,GGAL,USD,60min,2022-04-04 09:00:00,8.9836,9.1457,8.9755,9.0079,45436.0,NaN,NaN
1,GGAL,USD,60min,2022-04-04 10:00:00,9.0484,9.0809,8.9755,8.9917,49030.0,NaN,NaN
2,GGAL,USD,60min,2022-04-04 11:00:00,8.9836,8.9998,8.9268,8.9268,41129.0,NaN,NaN
3,GGAL,USD,60min,2022-04-04 12:00:00,8.9268,9.0160,8.9268,8.9512,34248.0,NaN,NaN
4,GGAL,USD,60min,2022-04-04 13:00:00,8.9593,8.9593,8.8863,8.9187,67152.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2101,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408
2102,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650
2103,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336
2104,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125


In [14]:
def set_test( df, intervalo, k, d, smth, dayM, semM):
        
        # Convert time
        #df["time"] = pd.to_datetime(df.index, utc=True)
        #df['timeArg'] = df['time'].dt.tz_convert('America/Argentina/Buenos_Aires')        
        #df['time'] = df['time'].dt.tz_convert(None)

        df = stochastic(df, "high", k, d, smth)
        df = stochastic(df, "med", k*dayM, d*dayM, smth*dayM)
        df = stochastic(df, "low", k*dayM*semM, d*dayM*semM, smth*dayM*semM)
    
        return df

dfstoch_hml= set_test(dfstoch, i , K, D, smoth, dayM , semM )
#%time set_test(dfstoch, i , K, D, smoth, dayM , semM ) 

In [35]:

def long_buy_crits( df):
    
    df["longbuylow"] = 0
    df["longbuymed"] = 0
    df["longbuyhigh"] = 0
    df["longselllow"] = 0
    df["longsellmed"] = 0
    df["longsellhigh"] = 0
    df["prices"] = 0.0
    df["state"] = ""    
    
    
    #for i in df.index: 
    for i in range(1, len(df)):
        
        # long buy  stochs criterias
        
        if df.loc[i,"klow"] > 20 and df.loc[i,"klow"] > df.loc[i,"dlow"] :
            df.loc[i, "longbuylow"] = 1
        else :
            df.loc[i, "longbuylow"] = 0
            
            
        if df.loc[i,"kmed"] > 20 and df.loc[i,"kmed"] > df.loc[i,"dmed"] :
            df.loc[i, "longbuymed"] = 1
        else :
            df.loc[i, "longbuymed"] = 0
            
    
        if df.loc[i,"khigh"] > 20 and df.loc[i,"khigh"] > df.loc[i,"dhigh"] :
            df.loc[i, "longbuyhigh"] = 1
        else :
            df.loc[i, "longbuyhigh"] = 0

        # long buy  states and prices          

        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i-1, "state"] == "" :        
            df.loc[i, "state"] = "buylong"            
            
        
        if  df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong" 

        # long sell state and price 

        if  df.loc[i, "longbuyhigh"] == 0 and df.loc[i, "longbuymed"] == 0 and   (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong")  : 
             df.loc[i, "state"] = "selllong"

        
         # Short buy stoch criterias
        
        if df.loc[i,"klow"] < 80 and  df.loc[i,"klow"] < df.loc[i,"dlow"] :
            df.loc[i, "longselllow"] = 1
        else :
            df.loc[i, "longselllow"] = 0
            
            
        if df.loc[i,"kmed"] < 80 and df.loc[i,"kmed"] < df.loc[i,"dmed"] :
            df.loc[i, "longsellmed"] = 1
        else :
            df.loc[i, "longsellmed"] = 0
            
    
        if df.loc[i,"khigh"] < 80 and df.loc[i,"khigh"] < df.loc[i,"dhigh"] :
            df.loc[i, "longsellhigh"] = 1
        else :
            df.loc[i, "longsellhigh"] = 0

            
    return df

df = long_buy_crits(dfstoch_hml) 

#%time long_buy_crits(dfstoch_hml)   
#display (df)

In [37]:

#df_filtrado = df[(df["buylong"] == 1)]  # & (df["longenter"] == 1)]
#
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # Mostra o conteúdo completo da
#display(df.head(500))
df_intervalo = df.iloc[840:1000]
display(df_intervalo)


,symbol,moeda,intervalo,datetime,open,high,low,close,volume,khigh,...,klow,dlow,longbuylow,longbuymed,longbuyhigh,longselllow,longsellmed,longsellhigh,state,prices
840,GGAL,USD,60min,2022-08-24 15:00:00,7.1317,7.2413,7.1317,7.1991,169881.0,80.216274,...,56.899187,34.323680,1,0,1,0,1,0,,0.0
841,GGAL,USD,60min,2022-08-24 16:00:00,7.2076,7.2076,7.2076,7.2076,11666.0,85.546558,...,57.087061,34.613242,1,0,1,0,1,0,,0.0
842,GGAL,USD,60min,2022-08-25 09:00:00,7.2244,7.2834,7.0052,7.0305,97477.0,79.852671,...,57.206129,34.901979,1,0,1,0,1,0,,0.0
843,GGAL,USD,60min,2022-08-25 10:00:00,7.0390,7.1401,6.9741,7.1321,120547.0,76.244682,...,57.326615,35.189773,1,0,0,0,1,1,,0.0
844,GGAL,USD,60min,2022-08-25 11:00:00,7.1486,7.2160,7.1015,7.1486,49391.0,72.070176,...,57.482063,35.476844,1,0,0,0,1,1,,0.0
845,GGAL,USD,60min,2022-08-25 12:00:00,7.1486,7.2076,7.1486,7.1738,21956.0,68.923165,...,57.617254,35.763071,1,0,0,0,1,1,,0.0
846,GGAL,USD,60min,2022-08-25 13:00:00,7.1907,7.2413,7.1631,7.1738,60670.0,65.402692,...,57.746119,36.048152,1,0,0,0,1,1,,0.0
847,GGAL,USD,60min,2022-08-25 14:00:00,7.1654,7.3087,7.1654,7.2750,103386.0,75.488894,...,57.911141,36.332298,1,0,1,0,1,0,,0.0
848,GGAL,USD,60min,2022-08-25 15:00:00,7.2708,7.3762,7.2708,7.3593,147072.0,81.756426,...,58.099427,36.615915,1,0,1,0,1,0,,0.0
849,GGAL,USD,60min,2022-08-25 16:00:00,7.3509,7.3509,7.3509,7.3509,17316.0,86.954724,...,58.294733,36.899034,1,1,1,0,0,0,buylong,0.0
